<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Model-E%C4%9Fitimi/ModelE%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
from datasets import load_dataset, DatasetDict

print("Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...")
# 'Yunij/kaggle-comp-daigt' veri seti yükleniyor.
raw_dataset = load_dataset('Yunij/kaggle-comp-daigt')

# Veri setinin bölümlerini kontrol edelim ve 'train' ile 'validation' olarak ayarlayalım.
# Eğer doğrudan 'train' ve 'test' bölümleri varsa bunları kullanırız.
# Eğer sadece 'train' varsa, onu böleriz.
if isinstance(raw_dataset, DatasetDict):
    if 'train' in raw_dataset and 'test' in raw_dataset:
        dataset = {
            'train': raw_dataset['train'],
            'validation': raw_dataset['test']
        }
    elif 'train' in raw_dataset:
        # Sadece 'train' bölümü varsa, bunu eğitim ve doğrulama olarak bölelim.
        print("Sadece 'train' bölümü bulundu, eğitim ve doğrulama olarak bölünüyor...")
        split_data = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)
        dataset = {
            'train': split_data['train'],
            'validation': split_data['test']
        }
    else:
        raise ValueError("Veri setinde 'train' bölümü bulunamadı.")
else:
    # Eğer raw_dataset bir DatasetDict değilse ve sadece bir Dataset ise (örneğin sadece train split)
    print("Yüklenen veri seti tek bir split içeriyor, eğitim ve doğrulama olarak bölünüyor...")
    split_data = raw_dataset.train_test_split(test_size=0.1, seed=42)
    dataset = {
        'train': split_data['train'],
        'validation': split_data['test']
    }

print("Yunij/kaggle-comp-daigt Veri Seti Başarıyla Yüklendi ve Bölümleri Ayarlandı!")
print("Eğitim seti boyutu:", len(dataset['train']))
print("Doğrulama seti boyutu:", len(dataset['validation']))
print("\nÖrnek bir veri (eğitim setinden):", dataset['train'][0])
print("\nÖrnek bir veri (doğrulama setinden):", dataset['validation'][0])

In [ ]:
from transformers import AutoTokenizer

# Model altyapısını hazır veri setine bağlama adımı
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fonksiyonu(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

# Hafızadaki yerel veri setimizin 'train' ve 'validation' bölümlerini tokenlaştırıyoruz
tokenized_datasets = {
    'train': dataset['train'].map(tokenize_fonksiyonu, batched=True),
    'validation': dataset['validation'].map(tokenize_fonksiyonu, batched=True)
}
print("Tokenlaştırma tamamlandı, model eğitimine hazır!")

In [ ]:
import torch
import numpy as np
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score

# 1. Ayarlar ve Yollar
device = "cuda" if torch.cuda.is_available() else "cpu"
drive_kayit_yolu = "/content/drive/MyDrive/en_iyi_detektor_modeli"
model_name = "distilbert-base-uncased"

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# 2. Kontrol Mekanizması
# NOT: Model tarafımızca önceden eğitilmiştir.
# Eğitim sürecini sıfırdan çalıştırmak isterseniz aşağıdaki değeri True yapabilirsiniz.
EGITIM_YAPILSIN_MI = False

if EGITIM_YAPILSIN_MI:
    print(f"--- Model bulunamadı veya zorunlu eğitim aktif. Eğitim başlıyor... ---")

    # Modeli sıfırdan yükle
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # Eğitim Ayarları
    training_args = TrainingArguments(
        output_dir="./ai_detector_local_results",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=10,
        report_to="none"
    )

    # Trainer Kurulumu
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
        compute_metrics=compute_metrics,
    )

    # Eğitimi Başlat
    trainer.train()

    # Kaydet
    os.makedirs(drive_kayit_yolu, exist_ok=True)
    trainer.save_model(drive_kayit_yolu)
    tokenizer.save_pretrained(drive_kayit_yolu)
    print(f"Eğitim tamamlandı ve model {drive_kayit_yolu} yoluna kaydedildi!")

else:
    print(f"--- Eğitim atlanıyor, Drive'daki hazır model yükleniyor... ---")

    tokenizer = AutoTokenizer.from_pretrained(drive_kayit_yolu)
    model = AutoModelForSequenceClassification.from_pretrained(drive_kayit_yolu).to(device)

model.eval()
print("Model şu an hafızada ve kullanıma hazır.")

--- Eğitim atlanıyor, Drive'daki hazır model yükleniyor... ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model şu an hafızada ve kullanıma hazır.


In [ ]:


import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Modelin Google Drive'daki klasör yolu
model_path = "/content/drive/MyDrive/en_iyi_detektor_modeli"

# Eğer Drive bağlı değilse otomatik bağlamaya çalışsın
if not os.path.exists("/content/drive"):
    print("Drive bağlı değil, bağlanılıyor...")
    from google.colab import drive
    drive.mount('/content/drive')

# Klasör kontrolü
if not os.path.exists(model_path):
    print(f"HATA: Google Drive'da '{model_path}' klasörü bulunamadı!")
    print("Lütfen önce modeli eğittiğinizden ve Drive'a kaydettiğinizden emin olun.")
else:
    print("Model Google Drive'dan yükleniyor, lütfen bekleyin...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model ve Tokenizer Yükleme
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval() # Modeli test moduna alıyoruz
    print("Model başarıyla yüklendi! Analize hazır.\n")

    def metni_analiz_et(metin):
        inputs = tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        olasiliklar = torch.softmax(logits, dim=-1)[0]
        tahmin_id = torch.argmax(logits, dim=-1).item()

        siniflar = {0: "İnsan Tarafından Yazılmış", 1: "Yapay Zeka Tarafından Yazılmış"}

        insan_skoru = olasiliklar[0].item() * 100
        yz_skoru = olasiliklar[1].item() * 100

        return siniflar[tahmin_id], insan_skoru, yz_skoru

    # --- Canlı Test Ekranı ---
    print("=== YAPAY ZEKA METİN DETEKTÖRÜ ===")
    print("Çıkış yapmak için küçük 'q' harfi yazıp Enter'a basın.")

    while True:
        kullanici_metni = input("\nAnaliz edilecek metni girin:\n> ")

        if kullanici_metni.strip().lower() == 'q':
            print("Program kapatıldı.")
            break

        if not kullanici_metni.strip():
            print("Lütfen boş bırakmayın.")
            continue

        karar, insan_yuzde, yz_yuzde = metni_analiz_et(kullanici_metni)

        print("\n" + "="*40)
        print(f"📊 SONUÇ: {karar}")
        print(f"👨‍💻 İnsan Yazısı İhtimali: %{insan_yuzde:.2f}")
        print(f"🤖 Yapay Zeka İhtimali: %{yz_yuzde:.2f}")
        print("="*40)


Model Google Drive'dan yükleniyor, lütfen bekleyin...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model başarıyla yüklendi! Analize hazır.

=== YAPAY ZEKA METİN DETEKTÖRÜ ===
Çıkış yapmak için küçük 'q' harfi yazıp Enter'a basın.

Analiz edilecek metni girin:
> q
Program kapatıldı.


In [ ]:
!pip install -q gradio
import gradio as gr

def arayuz_analiz(metin):
    karar, insan_yuzde, yz_yuzde = metni_analiz_et(metin)
    sonuc_metni = f"📊 TAHMİN: {karar}\n\n"
    sonuc_metni += f"👨‍💻 İnsan Yazısı: %{insan_yuzde:.2f}\n"
    sonuc_metni += f"🤖 Yapay Zeka: %{yz_yuzde:.2f}"
    return sonuc_metni

# Gradio Arayüzü Oluşturma
iface = gr.Interface(
    fn=arayuz_analiz,
    inputs=gr.Textbox(lines=5, placeholder="Analiz edilecek metni buraya yapıştırın...", label="Giriş Metni"),
    outputs=gr.Textbox(label="Analiz Sonucu"),
    title="AI Metin Detektörü",
    description="Bu model, girilen metnin bir insan tarafından mı yoksa yapay zeka tarafından mı yazıldığını tahmin eder.",
    theme="soft"
)

# share=True parametresi sayesinde hocanıza atabileceğiniz 72 saatlik geçici bir web linki oluşur
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://adcfe8fcac12c389e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
